In [2]:
!git clone https://github.com/rafiyamo/manga-ocr-translation.git
%cd manga-ocr-translation

Cloning into 'manga-ocr-translation'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 46 (delta 16), reused 30 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 14.44 KiB | 14.44 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/manga-ocr-translation


In [3]:
!pip install -r requirements.txt

# Install PyTorch with CUDA support (works on most Colab GPUs)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [4]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.9.0+cu126
CUDA available: True
GPU name: Tesla T4


In [5]:
from src.pipeline import process_page

# This will fail if there's no image at that path in Colab, that's okay;
# we're mainly checking that imports work.
print("Imports OK.")

Imports OK.


In [6]:
# ===== Config & basic imports =====

import math
from dataclasses import dataclass
from typing import List, Tuple, Dict

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Some basic hyperparameters (we can tune later)
@dataclass
class TrainConfig:
    max_epochs: int = 10
    batch_size: int = 32
    lr: float = 1e-3
    max_src_len: int = 64
    max_tgt_len: int = 64
    embed_dim: int = 128
    hidden_dim: int = 256

config = TrainConfig()
config

Using device: cuda


TrainConfig(max_epochs=10, batch_size=32, lr=0.001, max_src_len=64, max_tgt_len=64, embed_dim=128, hidden_dim=256)

In [7]:
# ===== Parallel dataset placeholder =====

from typing import NamedTuple


class ParallelExample(NamedTuple):
    src: str  # source sentence (e.g., Japanese)
    tgt: str  # target sentence (English)


def load_parallel_data() -> List[ParallelExample]:
    """
    Placeholder loader.

    Later:
      - This will read a CSV/TSV or text file containing sentence pairs.
      - For now we just return a tiny toy dataset so that the rest
        of the pipeline (tokenizer, model, training loop) can be developed.
    """
    examples = [
        ParallelExample("hello", "hello"),
        ParallelExample("good morning", "good morning"),
        ParallelExample("thank you", "thank you"),
        ParallelExample("how are you?", "how are you?"),
    ]
    return examples


toy_data = load_parallel_data()
len(toy_data), toy_data[:3]

(4,
 [ParallelExample(src='hello', tgt='hello'),
  ParallelExample(src='good morning', tgt='good morning'),
  ParallelExample(src='thank you', tgt='thank you')])

In [8]:
# ===== Tokenizer & vocabulary (character-level for now) =====

PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"


def build_char_vocab(examples: List[ParallelExample]) -> Dict[str, int]:
    """
    Build a character-level vocabulary from both source and target text.
    """
    chars = set()
    for ex in examples:
        chars.update(list(ex.src))
        chars.update(list(ex.tgt))

    # special tokens first
    vocab = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2}
    for ch in sorted(chars):
        if ch not in vocab:
            vocab[ch] = len(vocab)
    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    """
    Encode a string as a list of token ids, with SOS/EOS and padding.
    """
    ids = [vocab[SOS_TOKEN]]
    for ch in text:
        ids.append(vocab.get(ch, vocab[PAD_TOKEN]))
        if len(ids) >= max_len - 1:
            break
    ids.append(vocab[EOS_TOKEN])

    # pad or truncate
    if len(ids) < max_len:
        ids.extend([vocab[PAD_TOKEN]] * (max_len - len(ids)))
    else:
        ids = ids[:max_len]
    return ids


def decode_ids(ids: List[int], inv_vocab: Dict[int, str]) -> str:
    """
    Decode token ids back into a string, ignoring PAD and special tokens.
    """
    chars = []
    for i in ids:
        token = inv_vocab.get(i, "")
        if token in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN):
            continue
        chars.append(token)
    return "".join(chars)


In [9]:
# ===== PyTorch Dataset & DataLoader =====

class ParallelCharDataset(Dataset):
    def __init__(self, examples: List[ParallelExample], vocab: Dict[str, int], config: TrainConfig):
        self.examples = examples
        self.vocab = vocab
        self.config = config

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = encode_text(ex.src, self.vocab, self.config.max_src_len)
        tgt_ids = encode_text(ex.tgt, self.vocab, self.config.max_tgt_len)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


# Build vocab & dataset from toy data for now
vocab = build_char_vocab(toy_data)
inv_vocab = {i: tok for tok, i in vocab.items()}

dataset = ParallelCharDataset(toy_data, vocab, config)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_src, batch_tgt in loader:
    print("src batch shape:", batch_src.shape)
    print("tgt batch shape:", batch_tgt.shape)
    break


src batch shape: torch.Size([2, 64])
tgt batch shape: torch.Size([2, 64])


In [10]:
# ===== Seq2Seq model (character-level GRU) =====

class Encoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_idx: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        src: [batch, src_len]
        returns: hidden state [1, batch, hidden_dim]
        """
        embedded = self.embedding(src)      # [batch, src_len, embed_dim]
        _, hidden = self.gru(embedded)      # hidden: [1, batch, hidden_dim]
        return hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_idx: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_tokens: torch.Tensor, hidden: torch.Tensor):
        """
        input_tokens: [batch]  (one timestep)
        hidden: [1, batch, hidden_dim]
        returns:
          logits: [batch, vocab_size]
          hidden: [1, batch, hidden_dim]
        """
        embedded = self.embedding(input_tokens.unsqueeze(1))  # [batch, 1, embed_dim]
        output, hidden = self.gru(embedded, hidden)           # output: [batch, 1, hidden_dim]
        logits = self.fc_out(output.squeeze(1))               # [batch, vocab_size]
        return logits, hidden


class Seq2Seq(nn.Module):
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        pad_idx: int,
        sos_idx: int,
        eos_idx: int,
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.sos_idx = sos_idx
        self.eos_idx = eos_idx

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, teacher_forcing_ratio: float = 0.5):
        """
        src: [batch, src_len]
        tgt: [batch, tgt_len]  (includes <sos> at position 0)
        returns: outputs [batch, tgt_len, vocab_size]
        """
        batch_size, tgt_len = tgt.size()
        vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src.device)

        hidden = self.encoder(src)

        # first input to decoder is the <sos> token from tgt
        input_tok = tgt[:, 0]

        for t in range(1, tgt_len):
            logits, hidden = self.decoder(input_tok, hidden)
            outputs[:, t] = logits

            # decide whether to use teacher forcing
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = logits.argmax(dim=1)

            input_tok = tgt[:, t] if teacher_force else top1

        return outputs


In [11]:
# ===== Training utilities =====

pad_idx = vocab[PAD_TOKEN]
sos_idx = vocab[SOS_TOKEN]
eos_idx = vocab[EOS_TOKEN]

encoder = Encoder(
    vocab_size=len(vocab),
    embed_dim=config.embed_dim,
    hidden_dim=config.hidden_dim,
    pad_idx=pad_idx,
)

decoder = Decoder(
    vocab_size=len(vocab),
    embed_dim=config.embed_dim,
    hidden_dim=config.hidden_dim,
    pad_idx=pad_idx,
)

model = Seq2Seq(encoder, decoder, pad_idx, sos_idx, eos_idx).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)


def train_one_epoch(model, loader, optimizer, criterion, config: TrainConfig) -> float:
    model.train()
    total_loss = 0.0

    for batch_src, batch_tgt in loader:
        batch_src = batch_src.to(device)
        batch_tgt = batch_tgt.to(device)

        optimizer.zero_grad()

        outputs = model(batch_src, batch_tgt, teacher_forcing_ratio=0.5)
        # outputs: [batch, tgt_len, vocab_size]

        # we compute loss for t=1..end (skip t=0 which is <sos>)
        logits = outputs[:, 1:].reshape(-1, outputs.size(-1))   # [batch*(tgt_len-1), vocab_size]
        target = batch_tgt[:, 1:].reshape(-1)                   # [batch*(tgt_len-1)]

        loss = criterion(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# quick smoke test: run a couple of epochs on the toy data
for epoch in range(3):
    loss = train_one_epoch(model, loader, optimizer, criterion, config)
    print(f"Epoch {epoch+1}: loss = {loss:.4f}")


Epoch 1: loss = 3.0269
Epoch 2: loss = 2.8086
Epoch 3: loss = 2.6603


In [14]:
# ===== Inference helper (greedy decoding) =====

def translate_text_with_model(model, text: str, max_len: int = 64) -> str:
    model.eval()
    with torch.no_grad():
        src_ids = encode_text(text, vocab, config.max_src_len)
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)  # [1, src_len]

        hidden = model.encoder(src_tensor)

        input_tok = torch.tensor([sos_idx], dtype=torch.long, device=device)

        decoded_ids = [sos_idx]
        for _ in range(max_len - 1):
            logits, hidden = model.decoder(input_tok, hidden)
            next_tok = logits.argmax(dim=1)  # [1]
            token_id = next_tok.item()
            decoded_ids.append(token_id)

            if token_id == eos_idx:
                break

            input_tok = next_tok

        return decode_ids(decoded_ids, inv_vocab)


# Trying it on a toy example (toy dataset is basically identity)
test_sentence = "hello"
translated = translate_text_with_model(model, test_sentence)
print("Input:     ", test_sentence)
print("Predicted: ", translated)


Input:      hello
Predicted:  hoou
